# Thesis and project toolkit {#sec-toolkit}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/main/notebooks/a06-toolkit.ipynb)

The chapters teach one method at a time. A thesis, a dissertation or a consulting project needs all of them at once, in the right order, written up so that an examiner or a client can follow and repeat the work. This appendix is the practical kit for that: how to plan the project, a folder layout, code that documents and tests a model for you, what to report, and a checklist to finish with.

**Who it is for.** Students writing a thesis or dissertation with a system dynamics model, and analysts running a modelling project for a client. **Prerequisites.** The process of chapters 1 to 15; the code uses @sec-package. **Time.** Read it once at the start of a project and again at the end.

In [ ]:
# Setup: run this cell first. In Colab it installs loopbricks (ten seconds).
try:
    import loopbricks as lb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/"
                           "loopbricks.git@main#subdirectory=python"])
    import loopbricks as lb
import json, os, platform, tempfile
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

SEED = 6
rng = np.random.default_rng(SEED)
WORK = tempfile.mkdtemp()      # a scratch folder; use your own project folder instead
print("loopbricks", lb.__version__)

## 1. The shape of a project

Every system dynamics project passes through the same stages, though good projects loop back through them several times. The table maps each stage to the chapters that teach it and to what it should leave behind.

| Stage | Question | Chapters | Leaves behind |
|---|---|---|---|
| Problem | What behaviour is the problem, over what time, for whom? | 1, 2 | a reference mode chart and a one-paragraph problem statement |
| Dynamic hypothesis | Which feedback structure could produce it? | 2, 3 to 7 | a causal loop diagram, a boundary chart, rival hypotheses |
| Formulation | What are the stocks, flows and decision rules? | 3 to 7, @sec-package | a model file, an equation listing, a parameter table with sources |
| Testing | Is the code right, and is the model fit for purpose? | 8, 9 | a test file that runs, sensitivity results |
| Evidence | What do the data say about the parameters? | 8, 10 | a calibration, residual checks, uncertainty bands |
| Policy | Which actions work, and how robust are they? | 11 | a policy comparison across scenarios, a brief |
| Communication | What should decision makers hear? | 11 | a report, a one-page brief, perhaps a simulator |

For a six-month thesis a workable split is roughly: a month on the problem and hypothesis, two months building and testing the model, a month on data and calibration, a month on policy, and a month writing, with the writing started from the first week. The most common failure is spending four months building a large model and leaving no time to test it or use it.

**A good question.** A system dynamics question is about **behaviour over time** and what changes it. Check yours against these:

- It names a behaviour you can draw: "why has the waiting list doubled since 2014 despite extra funding?", not "the waiting-list system".
- It has a time horizon long enough for the feedback to play out, usually several times the longest delay.
- It has someone who can act on the answer, and levers they control.
- There is at least some data, or expert knowledge, on the reference mode and the key parameters.
- It is small enough to answer: a model of ten to thirty stocks answers most thesis questions; a model of three hundred answers none of them in six months.

## 2. A project folder

Keep everything for the project in one folder under version control (Git), so that any result can be traced to the code and data that made it. A layout that works:

| Folder or file | Holds |
|---|---|
| `README.md` | the question, how to run everything, and where each result comes from |
| `requirements.txt` | the Python packages, with versions |
| `data/raw/` | data exactly as received, never edited |
| `data/processed/` | data after cleaning, made by a script |
| `models/` | the model as a `loopbricks` spec in JSON, one file per version that matters |
| `notebooks/` | one notebook per stage: problem, model, tests, calibration, policy |
| `tests/` | test functions that run the whole model check (section 4) |
| `reports/figures/` | every figure in the thesis, saved by the notebooks |

The function below creates that skeleton, with a starting README.

In [ ]:
README = """# {title}

Question: {question}

How to run: `pip install -r requirements.txt`, then run the notebooks in
`notebooks/` in order. Every figure in `reports/figures/` is made by a notebook.

Data: raw files in `data/raw/` are never edited; `data/processed/` is made by code.
"""

def new_project(path, title, question):
    """Make the folder layout of a modelling project, with a starting README."""
    for sub in ["data/raw", "data/processed", "models", "notebooks", "tests",
                "reports/figures"]:
        os.makedirs(os.path.join(path, sub), exist_ok=True)   # no error if it exists
    with open(os.path.join(path, "README.md"), "w") as f:
        f.write(README.format(title=title, question=question))
    with open(os.path.join(path, "requirements.txt"), "w") as f:
        f.write(f"loopbricks=={lb.__version__}\nnumpy=={np.__version__}\n"
                f"pandas=={pd.__version__}\nmatplotlib=={matplotlib.__version__}\n")
    return sorted(os.path.relpath(os.path.join(d, x), path)
                  for d, dirs, files in os.walk(path) for x in dirs + files)

project = os.path.join(WORK, "waiting-list-thesis")
print("\n".join(new_project(project, "Waiting lists that push back",
                            "Why has the list doubled since 2014 despite extra funding?")))

`os.makedirs(..., exist_ok=True)` creates a folder and any missing parents, and does nothing if it is already there; `os.walk` visits every folder below a path. Run `new_project("my-thesis", ...)` once at the start, then `git init` in that folder.

**Save the model as a file.** A model built in a notebook should also live in `models/` as JSON, so the thesis, the tests and the LoopBricks app all use the same equations. `model.spec` is the dictionary behind any model, and `lb.Model.from_file` reads it back.

In [ ]:
spec = {
    "name": "Waiting list", "timeUnit": "month", "T": 120, "dt": 0.125,
    "stocks": [{"id": "wl", "name": "Waiting list", "init": "32000",
                "units": "patients", "nonNegative": True}],
    "flows": [
        {"id": "ref", "name": "Referrals", "to": "wl",
         "eq": "Referral_rate * (1 + Referral_growth) ^ TIME", "units": "patients/month"},
        {"id": "trt", "name": "Treatments", "from": "wl",
         "eq": "MIN(Capacity, Waiting_list / Minimum_treatment_time)",
         "units": "patients/month"},
    ],
    "vars": [
        {"id": "rr", "name": "Referral rate", "eq": "4000", "units": "patients/month",
         "source": "average of 2014 referrals, hospital records"},
        {"id": "rg", "name": "Referral growth", "eq": "0.0008", "units": "1/month",
         "source": "fitted, chapter 5"},
        {"id": "cap", "name": "Capacity", "eq": "3950", "units": "patients/month",
         "source": "clinic schedules, 2014"},
        {"id": "mtt", "name": "Minimum treatment time", "eq": "0.5", "units": "month",
         "source": "assumption: two weeks to book and see a patient"},
    ],
}
model = lb.Model.from_spec(spec)
path = os.path.join(project, "models", "waiting-list-v1.json")
with open(path, "w") as f:
    json.dump(model.spec, f, indent=1)           # the model as a readable file
again = lb.Model.from_file(path)
same = np.array_equal(model.run().to_numpy(), again.run().to_numpy())
print("saved and reloaded; identical runs:", same)

Each variable above carries a `source`: where its value comes from. The package ignores fields it does not use, so you can keep the evidence for every number next to the number itself, which is what the next section turns into a table.

## 3. Documentation that writes itself

Examiners and reviewers ask the same things of every model: what is in it, where each number comes from, and what was left out [@martinezmoyano2012; @rahmandad2012]. Writing those tables by hand guarantees they drift from the model. Generate them from the model instead.

In [ ]:
def model_table(model):
    """Every element of a model: kind, equation, units and source, as a DataFrame."""
    rows = []
    for kind, items in [("stock", model.stocks), ("flow", model.flows),
                        ("variable", model.vars)]:
        for e in items:
            rows.append({"name": e["name"], "kind": kind,
                         "equation or initial value": e.get("eq", e.get("init")),
                         "units": e.get("units", ""),
                         "source": e.get("source", "")})
    return pd.DataFrame(rows)

table = model_table(model)
print(table.to_string(index=False))
missing = table[(table.kind == "variable") & (table.source == "")]
print("\nvariables with no source:", list(missing.name) or "none")

`table.to_markdown()` or `table.to_latex()` writes the same table straight into a report (the first needs the `tabulate` package). Put the full table and `model.equations()` in an appendix of the thesis, and a short version in the main text.

Two more tables belong in every model report. The **boundary chart** (@sec-story-to-diagram) lists what the model makes endogenous, what it takes as exogenous, and what it deliberately excludes, with a reason for each exclusion. The **parameter table** gives, for each uncertain parameter, its value, its plausible range and the evidence; the range is what the sensitivity analysis and the calibration bounds use.

**Try it 3.**

- *Check.* Remove the `source` of `Capacity` from `spec`, rebuild the model and rerun the check. What does it report?
- *Apply.* Write a function that adds a `units ok` column, flagging rows whose units are empty.
- *Extend.* Generate the model table for `lb.molecule("stockmgmt")` and save it as a CSV file in `reports/`.

<details><summary>Worked answer</summary>

```python
spec["vars"][2].pop("source")                    # Capacity loses its source
print(model_table(lb.Model.from_spec(spec)).query("source == ''").name.tolist())
t = model_table(model)
t["units ok"] = t["units"] != ""
print(t[["name", "units ok"]].to_string(index=False))
model_table(lb.molecule("stockmgmt")).to_csv(
    os.path.join(project, "reports", "stockmgmt-table.csv"), index=False)
```

</details>

## 4. Tests that run themselves

Chapter 12 explains why a model needs tests; this is the kit to make them a habit. Write each check as a small function that fails loudly with `assert`, keep them in `tests/test_model.py`, and run them with `pytest` after every change. Here are four that suit almost any model.

In [ ]:
def test_conservation(model, stock="Waiting list"):
    """Every unit that came in either went out or is still there."""
    run = model.run(include="all", warn_clips=False)
    dt = run.index[1] - run.index[0]
    ins = [f["name"] for f in model.flows if f.get("to") == model._find(stock)["id"]]
    outs = [f["name"] for f in model.flows if f.get("from") == model._find(stock)["id"]]
    net = (run[ins].sum(axis=1) - run[outs].sum(axis=1)).iloc[:-1].sum() * dt
    change = run[stock].iloc[-1] - run[stock].iloc[0]
    assert abs(net - change) < 1e-6 * max(1, abs(change)), (net, change)

def test_extremes(model):
    """No capacity: nobody is treated. No referrals: the list drains, never below zero."""
    none_treated = model.run(capacity=0)
    assert (none_treated["Treatments"] == 0).all()
    empty = model.run(referral_rate=0, warn_clips=False)
    assert empty["Waiting list"].min() >= 0 and not empty.attrs["clipped"]

def test_time_step(model, stock="Waiting list", tolerance=0.001):
    """Halving dt changes the final value by less than the tolerance (a share)."""
    a = model.run()[stock].iloc[-1]
    b = model.run(dt=model.dt / 2)[stock].iloc[-1]
    assert abs(a - b) <= tolerance * abs(a), (a, b)

def test_reference_mode(model, stock="Waiting list"):
    """The model reproduces the direction of the problem: the list grows."""
    run = model.run()[stock]
    assert run.iloc[-1] > run.iloc[0] * 1.05

for test in [test_conservation, test_extremes, test_time_step, test_reference_mode]:
    test(model)
    print(f"passed: {test.__name__}")

`model._find(name)` is the package's internal look-up of an element by name; the test uses it to find which flows touch the stock. In a `tests/test_model.py` file, each function would take no argument and load the model from `models/` itself; `pytest` then finds every function whose name starts with `test_`, runs it and reports failures. Add a test whenever you fix a mistake, so that it cannot come back unnoticed.

**Try it 4.**

- *Check.* Break the model on purpose, for example `model.with_equation("Treatments", "Capacity")`, and see which test fails.
- *Apply.* Write `test_policy_sign`: more capacity must never give a longer list at the end.
- *Extend.* Turn the four tests into a file in `tests/` that loads `models/waiting-list-v1.json`, and run it with `pytest` from a terminal.

<details><summary>Worked answer</summary>

```python
broken = model.with_equation("Treatments", "Capacity")
try:
    test_extremes(broken)
except AssertionError:
    print("test_extremes caught it: with no referrals the list is held up by the floor")

def test_policy_sign(model):
    base = model.run()["Waiting list"].iloc[-1]
    more = model.run(capacity=model.params["capacity"] + 100)["Waiting list"].iloc[-1]
    assert more <= base
test_policy_sign(model)
```

</details>

## 5. Figures and results that can be repeated

Every number in the thesis should come from a run anyone can repeat. Four habits make that true:

- **Seed every random process** once, at the top of each notebook (`rng = np.random.default_rng(SEED)`), as every chapter of this book does.
- **Record the versions** of Python and the packages with the results (below).
- **Save every figure from code**, at a fixed size and resolution, into `reports/figures/`, never by copying from the screen.
- **Run the notebooks top to bottom** from a fresh kernel before you hand anything in.

In [ ]:
#| fig-alt: "The waiting list over ten years in the project's model, rising from 32,000, saved to the project's figures folder."
def save_figure(fig, name, folder=os.path.join(project, "reports", "figures")):
    """Save a figure as PNG for drafts and PDF for print, at a fixed resolution."""
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(folder, f"{name}.{ext}"), dpi=300, bbox_inches="tight")

run = model.run()
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(run.index / 12, run["Waiting list"] / 1000)
ax.set(xlabel="time (years)", ylabel="patients waiting (thousands)")
save_figure(fig, "fig-1-baseline")
plt.show()

versions = {"python": platform.python_version(), "loopbricks": lb.__version__,
            "numpy": np.__version__, "pandas": pd.__version__}
print(versions)
print(sorted(os.listdir(os.path.join(project, "reports", "figures"))))

Figure: the project's baseline run, saved to the figures folder as PNG and PDF. The same function saves every figure, so all of them share a size and resolution.

`bbox_inches="tight"` trims the white margin; `dpi=300` is the usual resolution for print. Put the version dictionary in the thesis appendix, or write it to a file next to the results.

## 6. Writing it up

A model-based thesis chapter or report usually follows the stages of section 1. What examiners look for in each part:

| Part | Must show |
|---|---|
| Problem | the reference mode as a chart, the time horizon, who the model is for |
| Hypothesis | the causal loop diagram, the boundary chart, the rival explanations considered |
| Model | the stock-and-flow structure, key equations in the text, all equations in an appendix, the parameter table with sources |
| Tests | the tests run and their results, including the ones that found mistakes |
| Fit and uncertainty | the fit to data with its error measures in data units, residual checks, uncertainty in parameters and results |
| Policy | the policies compared, the measures, the scenarios, why the recommended one is robust |
| Limitations | what the boundary leaves out, which conclusions depend on uncertain assumptions |

The reporting guidelines of @rahmandad2012 give a fuller checklist for simulation studies, and @martinezmoyano2012 sets out what model documentation needs for others to reuse a model. Chapter 15's **brief** is the format for decision makers: the problem in one sentence, the finding, the recommendation, how sure we are, and what would change the answer.

**Data and ethics.** Check the licence of every data set before you publish results from it, and say in the report where each came from. Label synthetic data as synthetic everywhere it appears, as this book does. Patient, staff or customer data must be anonymised or aggregated, and interviews and group model building sessions need consent, usually through your institution's ethics process before you start.

## 7. Mistakes examiners notice

- A model with no reference mode, so there is nothing to explain and no way to test it.
- A causal loop diagram presented as the result, with no simulation to show the loops produce the behaviour.
- Units that do not balance, or a time step that changes the results.
- A beautiful fit to history taken as proof the model is right (@sec-learning, @sec-calibration).
- Parameters with no source, or calibrated values outside any plausible range.
- Policies tested only in the base case, with no sensitivity or scenarios.
- Results that cannot be reproduced: no code, no seed, figures copied by hand.

## 8. Checklist before you hand in

| Done | Item |
|---|---|
| | The question names a behaviour over time and someone who can act |
| | The reference mode is shown, with its data source |
| | The causal loop diagram and boundary chart are in the report |
| | Every equation is in an appendix, generated from the model file |
| | Every parameter has a value, a range, units and a source |
| | The tests (conservation, extremes, time step, reference mode) pass |
| | Sensitivity analysis covers the uncertain parameters that matter |
| | Fit is reported in data units, with residual checks and uncertainty |
| | Policies are compared across scenarios, with costs where relevant |
| | Limitations are stated, including what the boundary excludes |
| | Every figure and number comes from a notebook that runs top to bottom |
| | Data licences, anonymisation and ethics approval are in order |
| | The model file, code and data are archived where others can find them |

## Further reading

- @rahmandad2012 on reporting guidelines for simulation-based research.
- @martinezmoyano2012 on documenting models so that others can understand and reuse them.
- @martinezmoyano2013 on best practices across the modelling process.
- @sterman2000 on the modelling process, from problem articulation to policy design.